# 05 — Evaluate adv-NO random-window rollout with Gaussian-circle viz (pool12 adv-NO k1)

Rollout eval for the model trained in [04_Sh_train_k1_random_12_l1+advno_1.ipynb](../Shalow/04_Sh_train_k1_random_12_l1+advno_1.ipynb).

**Model:** `shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_pool12_advnorandom_k1`
- **gUnet**, dropout 0.1, **adv-NO** (L1 + perceptual + adversarial, full-field loss)
- **Window pool** training: 12 cached crops per anchor (`pool12_advnorandom_k1`)
- Train export: `Shalow/data/random_win_128/sample_13_22_uv_dt360`

**Eval export:** `random_win_128` → 2023 test `sample_23_uv_dt360`.

**Checkpoint:** `Shalow/Models/adv_no/..._pool12_advnorandom_k1/best_model.pth` + `norm_stats.pt`

3-step autoregressive rollout at **+15d / +30d / +45d**. GT +15d from export; +30d/+45d from Zarr (per-sample random crop).

**Rollout viz** (`samples_gauss_circles/`): normalized increments Δu,Δv with Gaussian-circle layout (σ=32, r≤16/32/64).

**Outputs:** `Shalow/Test&Rollout/..._pool12_advnorandom_k1__on__random_win_128/`


In [ ]:
import json
import os
import sys
from pathlib import Path

import torch

NOTEBOOK_DIR = Path.cwd().resolve()
MW_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Magic_window" else NOTEBOOK_DIR / "Magic_window"
SHALOW_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Shalow" else NOTEBOOK_DIR.parent / "Shalow"
if not SHALOW_DIR.is_dir():
    SHALOW_DIR = MW_DIR.parent / "Shalow"
ROOT = SHALOW_DIR.parent
os.chdir(SHALOW_DIR)
for _p in (str(ROOT), str(SHALOW_DIR)):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

DATA_DIR = SHALOW_DIR / "data"
from window_paths import test_prefix
import importlib
import run_registry as _run_registry
importlib.reload(_run_registry)
from run_registry import (
    RUN_TAG,
    resolve_model_checkpoint,
    rollout_eval_name,
    save_rollout_results,
    run_dir,
    run_name,
    test_rollout_dir,
    test_rollout_dir_with_model,
    update_comparison_index,
    weight_decay_run_tag,
    generator_run_tag,
    dropout_run_tag,
)
MODELS_DIR = SHALOW_DIR / "Models" / "adv_no"

# Must match 04_Sh_train_k1_random_12_l1+advno_1.ipynb
WEIGHT_DECAY = 1e-3
VAL_SPLIT_MODE = "random"
N_WINDOWS_PER_SAMPLE = 12
USE_WINDOW_POOL = True
DYNAMIC_RANDOM_CROP = True
TRAIN_CROPS_PER_ANCHOR = 1
LOSS_MODE = "adv_no"
LOSS_SPATIAL_MODE = "full"
_val_suffix = "chrono" if VAL_SPLIT_MODE == "chronological" else VAL_SPLIT_MODE
_crops_tag = f"_k{int(TRAIN_CROPS_PER_ANCHOR)}" if int(TRAIN_CROPS_PER_ANCHOR) != 2 else ""
_crop_prefix = f"pool{N_WINDOWS_PER_SAMPLE}" if USE_WINDOW_POOL else ("dynwin" if DYNAMIC_RANDOM_CROP else "fixwin")
_loss_tag = "advno" if LOSS_MODE == "adv_no" else "l1"
EXPERIMENT_TAG = f"{_crop_prefix}_{_loss_tag}{_val_suffix}{_crops_tag}"

GENERATOR = "gUnet"
UNET_DROPOUT = 0.1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RUN_TAG = weight_decay_run_tag(RUN_TAG, WEIGHT_DECAY)
RUN_TAG = dropout_run_tag(RUN_TAG, UNET_DROPOUT)
RUN_TAG = generator_run_tag(RUN_TAG, GENERATOR)
RUN_TAG = f"{RUN_TAG}_{EXPERIMENT_TAG}"

EXPORT_KEY = "random_win_128"
EVAL_DATA_PREFIX = None

MODEL_RUN_DIR = SHALOW_DIR / (
    "Models/adv_no/"
    "shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_pool12_advnorandom_k1"
)

DT = 360
if EVAL_DATA_PREFIX is not None:
    DATA_PREFIX = str(Path(EVAL_DATA_PREFIX).resolve())
    if not Path(f"{DATA_PREFIX}_sims.npy").exists():
        raise FileNotFoundError(f"Missing eval export: {DATA_PREFIX}_sims.npy")
else:
    DATA_PREFIX = test_prefix(DATA_DIR, EXPORT_KEY, dt=DT)

USE_SHARED_MODEL = MODEL_RUN_DIR is not None
MODEL_RUN_DIR = Path(MODEL_RUN_DIR).resolve()
(
    train_cfg,
    MODEL_PATH,
    STATS_PATH,
    BEST_EPOCH,
    BEST_VAL_LOSS,
    MODEL_RUN_NAME,
) = resolve_model_checkpoint(MODEL_RUN_DIR)
RUN_NAME = rollout_eval_name(MODEL_RUN_NAME, EXPORT_KEY)
RUN_DIR = MODEL_RUN_DIR
TEST_ROLLOUT_DIR = test_rollout_dir_with_model(SHALOW_DIR, EXPORT_KEY, MODEL_RUN_NAME)

_expected_train = {
    "dynamic_random_crop": DYNAMIC_RANDOM_CROP,
    "train_crops_per_anchor": TRAIN_CROPS_PER_ANCHOR,
    "val_split_mode": VAL_SPLIT_MODE,
    "use_window_pool": USE_WINDOW_POOL,
    "loss_mode": LOSS_MODE,
    "loss_spatial_mode": LOSS_SPATIAL_MODE,
}
for _key, _want in _expected_train.items():
    _got = train_cfg.get(_key)
    if _got != _want:
        raise ValueError(
            f"Checkpoint {_key}={_got!r} but setup expects {_want!r}. "
            f"Update flags or MODEL_RUN_DIR."
        )

SAMPLES_DIR = TEST_ROLLOUT_DIR / "samples"
SAMPLES_GAUSS_DIR = TEST_ROLLOUT_DIR / "samples_gauss_circles"
for _d in (TEST_ROLLOUT_DIR, SAMPLES_DIR, SAMPLES_GAUSS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

DT = int(train_cfg["dt_hours"])
iD = int(train_cfg["iD"])
UNET_DIM = int(train_cfg.get("unet_dim", 32))
UNET_DIM_MULTS = tuple(train_cfg.get("unet_dim_mults", (1, 2, 4, 8)))

GAUSS_SIGMA = float(train_cfg.get("loss_gaussian_sigma", 32.0))
GAUSS_CIRCLE_RADII = tuple(train_cfg.get("viz_gaussian_circle_radii", (16.0, 32.0, 64.0)))
GAUSS_INNER_RADII = tuple(train_cfg.get("viz_gaussian_inner_circle_radii", (32.0, 16.0)))
GAUSS_FIG_SCALE = float(train_cfg.get("viz_gaussian_fig_scale", 2.6))
VIZ_RESIDUAL = str(train_cfg.get("viz_residual_metric", "l1"))
PLOT_N_SAMPLES = 5
PLOT_SAMPLE_SEED = 42
_cfg_generator = str(train_cfg.get("generator", "gUnet"))
GENERATOR = _cfg_generator
UNET_DROPOUT = float(train_cfg.get("unet_dropout", UNET_DROPOUT))

print("Working directory:", Path.cwd())
print("SHALOW_DIR:", SHALOW_DIR)
print("experiment tag:", EXPERIMENT_TAG)
print("EXPORT_KEY:", EXPORT_KEY)
print("loss_mode:", train_cfg.get("loss_mode"))
print("loss_spatial_mode:", train_cfg.get("loss_spatial_mode"))
print("generator:", GENERATOR, "dropout:", UNET_DROPOUT)
print("device:", DEVICE)
print("Model run:", MODEL_RUN_NAME)
print("Rollout label:", RUN_NAME)
print("Test&Rollout dir:", TEST_ROLLOUT_DIR)
print("Best checkpoint:", MODEL_PATH)
if BEST_EPOCH:
    print(f"  best val L1 epoch {BEST_EPOCH}: {BEST_VAL_LOSS:.6f}")
print("Gaussian plots:", SAMPLES_GAUSS_DIR)
print("Eval export:", DATA_PREFIX)


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from models.build import import_unet2d

from src.train.fix_window_viz import (
    _uv_with_speed,
    normalize_physical_uv,
    normalized_uv_increment,
    rollout_pred_increment,
    save_gaussian_circles_sample_figure,
)

Unet2D = import_unet2d(GENERATOR)  # gUnet | shalow_gUnet | shalow_gUnet_xs

from src.data.dataloaders.guiana_margem_tensor import GuianaDataloader
from src.data.useful_transforms import FixedCrop, RotatedFixedCrop

# RUN_NAME, MODEL_PATH, STATS_PATH, DT, iD, UNET_* from setup cell above
DT_STEP_DAYS = DT // 24
CHANNEL_NAMES = ["U", "V"]

SPEED_PATCH_SIZE = 4  # center patch for plots + center metrics
EVAL_HORIZONS = (DT_STEP_DAYS, 2 * DT_STEP_DAYS, 3 * DT_STEP_DAYS)
H1, H2, H3 = EVAL_HORIZONS
FORECAST_TITLES = tuple(f"t[n+{d}d]" for d in EVAL_HORIZONS)

EXPORT_CONFIG_PATH = Path(f"{DATA_PREFIX}_export_config.json")
WINDOW_META = np.load(f"{DATA_PREFIX}_window.npy")
_WINDOWS_PATH = Path(f"{DATA_PREFIX}_windows.npy")
USE_RANDOM_WINDOWS = _WINDOWS_PATH.exists() and int(WINDOW_META[0]) < 0

# ----------------------------
# Load test data
# ----------------------------
def load_test_data(
    sims_path: str = f"{DATA_PREFIX}_sims.npy",
    mask_path: str = f"{DATA_PREFIX}_mask.npy",
    day_path: str = f"{DATA_PREFIX}_day.npy",
    hour_path: str = f"{DATA_PREFIX}_hour.npy",
    latlon_path: str = f"{DATA_PREFIX}_latlon.npy",
    indices_path: str = f"{DATA_PREFIX}_indices.npy",
    windows_path: str = f"{DATA_PREFIX}_windows.npy",
    stats_path: str = str(STATS_PATH),
    test_split: float = 1.0,
    seed: int = 42,
):
    sims = torch.from_numpy(np.load(sims_path))
    mask = torch.from_numpy(np.load(mask_path))
    day = torch.from_numpy(np.load(day_path))
    hour = torch.from_numpy(np.load(hour_path))
    latlon = torch.from_numpy(np.load(latlon_path))
    export_indices = np.load(indices_path)
    export_windows = np.load(windows_path) if Path(windows_path).exists() else None

    # Coastal magic window: keep all sequences; static land mask per-pixel (matches training).
    n = len(sims)
    land_frac = float(mask[0].mean()) if n else 0.0
    static_mask = bool(n) and torch.all(mask == mask[0:1])
    print(
        f"Loaded {n} sequences | land {land_frac:.1%}"
        + (" | static coastal mask" if static_mask else "")
    )
    n_test = int(n * test_split)
    if test_split >= 1.0:
        test_idx = np.arange(n)
    else:
        rng = np.random.default_rng(seed)
        test_idx = rng.permutation(n)[:n_test]

    stats = torch.load(stats_path, map_location="cpu")
    mean = stats["mean"]
    std = stats["std"]
    mean_x = mean.repeat(1, 3, 1, 1)
    std_x = std.repeat(1, 3, 1, 1)

    return {
        "sims": sims[test_idx],
        "mask": mask[test_idx],
        "day": day[test_idx] / 365.0,
        "hour": hour[test_idx] / 365.0,
        "latlon": latlon[test_idx],
        "mean": mean_x,
        "std": std_x,
        "y_mean": mean,
        "y_std": std,
        "n_frames": int(sims.shape[1]),
        "export_indices": export_indices[test_idx],
        "export_windows": export_windows[test_idx] if export_windows is not None else None,
    }


def _build_zarr_loader(*, use_random_windows: bool = False):
    cfg = json.loads(EXPORT_CONFIG_PATH.read_text())
    transform = None
    if not use_random_windows:
        top, left, h, w, angle = WINDOW_META
        if float(angle) == 0.0:
            crop = FixedCrop(size=(int(h), int(w)), top=int(top), left=int(left))
        else:
            crop = RotatedFixedCrop(
                size=(int(h), int(w)),
                top=int(top),
                left=int(left),
                angle_deg=float(angle),
            )
        transform = crop
    return GuianaDataloader(
        path=cfg["zarr_path"],
        dt=DT,
        conditioning_steps=3,
        forecast_steps=1,
        transform=transform,
    )


def _load_cropped_uv_at(
    zarr_ds,
    timestamp,
    *,
    top: int | None = None,
    left: int | None = None,
    crop_h: int | None = None,
    crop_w: int | None = None,
) -> torch.Tensor:
    """Load cropped u,v at one Zarr time. Returns [2, H, W]."""
    t = pd.Timestamp(timestamp)
    fields = zarr_ds.load_uvssh_frame(t).astype(np.float32, copy=False)  # [3, H, W] u,v,ssh
    fields = np.nan_to_num(fields, nan=0.0)

    if top is not None and left is not None:
        crop_h = int(crop_h)
        crop_w = int(crop_w)
        patch = fields[:2, top : top + crop_h, left : left + crop_w]
        return torch.from_numpy(patch.copy())

    h, w = fields.shape[-2:]
    mask_ch = (~np.isfinite(fields[:1])).astype(np.float32)
    bat = np.ones((1, h, w), dtype=np.float32)
    lat, lon = zarr_ds._latlon_grid(h, w)
    bundle = torch.cat(
        [
            torch.tensor(mask_ch, dtype=torch.float32).unsqueeze(0),
            torch.tensor(fields, dtype=torch.float32).unsqueeze(0),
            torch.tensor(bat, dtype=torch.float32).unsqueeze(0),
            torch.tensor(lat, dtype=torch.float32).unsqueeze(0).unsqueeze(0),
            torch.tensor(lon, dtype=torch.float32).unsqueeze(0).unsqueeze(0),
        ],
        dim=1,
    )
    if zarr_ds.transform is not None:
        bundle = zarr_ds.transform(bundle)
        if not torch.is_tensor(bundle):
            bundle = torch.tensor(bundle, dtype=torch.float32)
    _, data, _, _, _ = torch.split(bundle, [1, 3, 1, 1, 1], dim=1)
    return data[0, :2].clone()


def load_gt_horizons_from_zarr(
    export_indices: np.ndarray,
    horizons_days: tuple[int, ...],
    export_windows: np.ndarray | None = None,
    *,
    crop_h: int | None = None,
    crop_w: int | None = None,
) -> dict[int, torch.Tensor]:
    """Load cropped GT u,v at anchor + each horizon (days). Returns {days: [N,2,H,W]}."""
    use_random = export_windows is not None
    zarr_ds = _build_zarr_loader(use_random_windows=use_random)
    if crop_h is None or crop_w is None:
        if use_random:
            crop_h = int(export_windows[0, 2])
            crop_w = int(export_windows[0, 3])
        else:
            _, _, crop_h, crop_w, _ = WINDOW_META
            crop_h, crop_w = int(crop_h), int(crop_w)

    out = {d: [] for d in horizons_days}
    missing = {d: 0 for d in horizons_days}
    for i, ds_idx in enumerate(export_indices):
        times = zarr_ds.sample_times(int(ds_idx))
        anchor = pd.Timestamp(times[2])
        win_top = win_left = None
        if use_random:
            win_top, win_left = int(export_windows[i, 0]), int(export_windows[i, 1])
        for d in horizons_days:
            t = anchor + pd.Timedelta(days=d)
            if t not in zarr_ds.time_to_idx:
                missing[d] += 1
                out[d].append(torch.zeros(2, crop_h, crop_w))
                continue
            out[d].append(
                _load_cropped_uv_at(
                    zarr_ds,
                    t,
                    top=win_top,
                    left=win_left,
                    crop_h=crop_h,
                    crop_w=crop_w,
                )
            )
    for d in horizons_days:
        if missing[d]:
            print(f"Warning: {missing[d]} samples missing Zarr GT at +{d}d")
        out[d] = torch.stack(out[d], dim=0)
    return out


# ----------------------------
# Center-patch metrics (4×4 at crop center)
# ----------------------------
def _center_slices(h: int, w: int, patch: int) -> tuple[slice, slice]:
    cy, cx = h // 2, w // 2
    half = patch // 2
    if patch % 2 == 0:
        return slice(cy - half, cy + half), slice(cx - half, cx + half)
    return slice(cy - half, cy + half + 1), slice(cx - half, cx + half + 1)


def _crop_metric_region(field: np.ndarray, mask: np.ndarray, patch_size: int | None):
    if patch_size is None:
        return field, mask
    sy, sx = _center_slices(field.shape[-2], field.shape[-1], patch_size)
    return field[..., sy, sx], mask[..., sy, sx]


def patch_rmse_2d(a: np.ndarray, b: np.ndarray, mask: np.ndarray, patch_size: int | None = None) -> float:
    patch_size = SPEED_PATCH_SIZE if patch_size is None else patch_size
    a, mask = _crop_metric_region(a, mask, patch_size)
    b, _ = _crop_metric_region(b, mask, patch_size)
    valid = mask == 0
    if not np.any(valid):
        return float("nan")
    return float(np.sqrt(np.mean((a[valid] - b[valid]) ** 2)))


# ----------------------------
# Inference
# ----------------------------
def _ocean_weight(mask: torch.Tensor, like: torch.Tensor) -> torch.Tensor:
    """[B, 1, H, W] ocean weights for Guiana-style land zeroing (matches training)."""
    return (mask == 0).to(dtype=like.dtype, device=like.device).unsqueeze(1)


def _zero_land(x: torch.Tensor, ocean: torch.Tensor) -> torch.Tensor:
    return x * ocean


def run_inference(
    model: torch.nn.Module,
    test_data: dict,
    device: str = "cuda:7",
    batch_size: int = 16,
    iD: int = 2,
    dt_step_days: float = DT_STEP_DAYS,
):
    model = model.to(device).eval()

    sims = test_data["sims"][:, :, :iD, :, :]
    day = test_data["day"]
    hour = test_data["hour"]
    mean = test_data["mean"]
    std = test_data["std"]
    masks = test_data["mask"]
    n_t = int(sims.shape[1])

    use_compact = n_t <= 4
    if use_compact:
        x1 = torch.cat([sims[:, 0], sims[:, 1], sims[:, 2]], dim=1).float()
    else:
        x1 = torch.cat([sims[:, 1], sims[:, 2], sims[:, 3]], dim=1).float()
    x1 = (x1 - mean) / (std + 1e-8)

    dataset = torch.utils.data.TensorDataset(x1, day, hour, masks)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    preds1, preds2, preds3 = [], [], []

    with torch.no_grad():
        for x_batch, d_batch, h_batch, m_batch in dataloader:
            x_batch = x_batch.to(device)
            d_batch = d_batch.to(device)
            h_batch = h_batch.to(device)
            m_batch = m_batch.to(device)
            ocean = _ocean_weight(m_batch, x_batch)
            x_batch = _zero_land(x_batch, ocean)

            inc1 = model(x_batch, d_batch + h_batch / 365.0)
            inc1 = _zero_land(inc1, ocean)
            pred1 = _zero_land(inc1 + x_batch[:, -iD:], ocean)

            x_batch2 = torch.cat([x_batch[:, iD : 2 * iD], x_batch[:, 2 * iD : 3 * iD], pred1], dim=1)
            x_batch2 = _zero_land(x_batch2, ocean)
            inc2 = model(x_batch2, d_batch + h_batch / 365.0 + dt_step_days / 365.0)
            inc2 = _zero_land(inc2, ocean)
            pred2 = _zero_land(inc2 + x_batch2[:, -iD:], ocean)

            x_batch3 = torch.cat([x_batch[:, 2 * iD : 3 * iD], x_batch2[:, -iD:], pred2], dim=1)
            x_batch3 = _zero_land(x_batch3, ocean)
            inc3 = model(x_batch3, d_batch + h_batch / 365.0 + 2 * dt_step_days / 365.0)
            inc3 = _zero_land(inc3, ocean)
            pred3 = _zero_land(inc3 + pred2, ocean)

            preds1.append(pred1.cpu())
            preds2.append(pred2.cpu())
            preds3.append(pred3.cpu())

    preds1 = torch.cat(preds1, dim=0)
    preds2 = torch.cat(preds2, dim=0)
    preds3 = torch.cat(preds3, dim=0)

    y_mean = test_data["y_mean"][:, :iD]
    y_std = test_data["y_std"][:, :iD]

    preds_denorm1 = preds1 * (y_std + 1e-8) + y_mean
    preds_denorm2 = preds2 * (y_std + 1e-8) + y_mean
    preds_denorm3 = preds3 * (y_std + 1e-8) + y_mean

    if use_compact:
        s1, s2, s3, s4 = sims[:, 0], sims[:, 1], sims[:, 2], sims[:, 3]
        gt_extra = load_gt_horizons_from_zarr(
            test_data["export_indices"],
            (H2, H3),
            export_windows=test_data.get("export_windows"),
            crop_h=int(sims.shape[-2]),
            crop_w=int(sims.shape[-1]),
        )
        s5, s6 = gt_extra[H2], gt_extra[H3]
    else:
        s1, s2, s3 = sims[:, 1], sims[:, 2], sims[:, 3]
        s4, s5, s6 = sims[:, 4], sims[:, 5], sims[:, 6]

    # GT increments: normalized (next − prev) / std (matches SimulationDataset / val viz)
    inc_gt1 = normalized_uv_increment(s4, s3, y_std, masks)
    inc_gt2 = normalized_uv_increment(s5, s4, y_std, masks)
    inc_gt3 = normalized_uv_increment(s6, s5, y_std, masks)

    # Pred increments: predicted state minus normalized GT anchor at the same time
    # (+15d: anchor s3; +30d: anchor s4; +45d: anchor s5)
    inc_pred1 = rollout_pred_increment(preds1, s3, y_mean, y_std, masks)
    inc_pred2 = rollout_pred_increment(preds2, s4, y_mean, y_std, masks)
    inc_pred3 = rollout_pred_increment(preds3, s5, y_mean, y_std, masks)

    return (
        s1, s2, s3, s4, s5, s6,
        preds_denorm1, preds_denorm2, preds_denorm3,
        inc_pred1, inc_pred2, inc_pred3,
        inc_gt1, inc_gt2, inc_gt3,
    )


def apply_mask(field: np.ndarray, mask: np.ndarray) -> np.ndarray:
    out = field.copy()
    out[mask == 1] = np.nan
    return out


def _symmetric_err_lim(fields: list[np.ndarray]) -> float:
    finite = np.concatenate([np.abs(f[np.isfinite(f)]) for f in fields if np.any(np.isfinite(f))])
    if finite.size == 0:
        return 1.0
    return max(float(np.nanmax(finite)), 1e-6)


def isotropic_spectrum(field):
    H, W = field.shape
    fft = np.fft.rfft2(field)
    power = np.abs(fft) ** 2
    ky = np.fft.fftfreq(H)
    kx = np.fft.rfftfreq(W)
    ky, kx = np.meshgrid(ky, kx, indexing="ij")
    k_mag = np.sqrt(kx ** 2 + ky ** 2)
    k_flat = k_mag.flatten()
    p_flat = power.flatten()
    k_max = min(H, W) // 2
    spectrum = np.zeros(k_max)
    for i in range(k_max - 1):
        mask_k = (k_flat >= i / k_max) & (k_flat < (i + 1) / k_max)
        if np.any(mask_k):
            spectrum[i] = p_flat[mask_k].mean()
    return np.arange(k_max), spectrum


def plot_sample(
    s1, s2, s3, s4, s5, s6, p1, p2, p3,
    test_data: dict,
    sample_idx: int = 0,
    save_path: str | None = None,
):
    s1 = s1[sample_idx].numpy()
    s2 = s2[sample_idx].numpy()
    s3 = s3[sample_idx].numpy()
    s4 = s4[sample_idx].numpy()
    s5 = s5[sample_idx].numpy()
    s6 = s6[sample_idx].numpy()
    p1 = p1[sample_idx].numpy()
    p2 = p2[sample_idx].numpy()
    p3 = p3[sample_idx].numpy()
    mask = test_data["mask"][sample_idx].numpy()
    day = test_data["day"][sample_idx].item()
    hour = test_data["hour"][sample_idx].item()
    lat, lon = test_data["latlon"][sample_idx].tolist()
    n_channels = p1.shape[0]

    S1 = (s1[0] ** 2 + s1[1] ** 2) ** 0.5
    S2 = (s2[0] ** 2 + s2[1] ** 2) ** 0.5
    S3 = (s3[0] ** 2 + s3[1] ** 2) ** 0.5
    gt_speeds = [
        (s4[0] ** 2 + s4[1] ** 2) ** 0.5,
        (s5[0] ** 2 + s5[1] ** 2) ** 0.5,
        (s6[0] ** 2 + s6[1] ** 2) ** 0.5,
    ]
    pred_speeds = [
        (p1[0] ** 2 + p1[1] ** 2) ** 0.5,
        (p2[0] ** 2 + p2[1] ** 2) ** 0.5,
        (p3[0] ** 2 + p3[1] ** 2) ** 0.5,
    ]
    errs = [
        (np.sqrt(np.nanmean((gt_speeds[i] - S3) ** 2)), np.sqrt(np.nanmean((pred_speeds[i] - gt_speeds[i]) ** 2)))
        for i in range(3)
    ]

    if save_path is not None:
        gt_fields = [s4, s5, s6]
        pred_fields = [p1, p2, p3]
        titles = FORECAST_TITLES
        gt_speeds_masked = [apply_mask(gs, mask) for gs in gt_speeds]
        pred_speeds_masked = [apply_mask(ps, mask) for ps in pred_speeds]
        speed_horizon_pairs = [
            (g, p, np.nanmin([g, p]), np.nanmax([g, p]))
            for g, p in zip(gt_speeds_masked, pred_speeds_masked)
        ]
        err_u = [apply_mask(pf[0] - gf[0], mask) for gf, pf in zip(gt_fields, pred_fields)]
        err_v = [apply_mask(pf[1] - gf[1], mask) for gf, pf in zip(gt_fields, pred_fields)]
        err_spd = [apply_mask(ps - gs, mask) for ps, gs in zip(pred_speeds, gt_speeds)]
        err_lim_u = _symmetric_err_lim(err_u)
        err_lim_v = _symmetric_err_lim(err_v)
        err_lim_spd = _symmetric_err_lim(err_spd)

        n_uv_rows = n_channels * 2
        speed_row_gt = n_uv_rows
        speed_row_pred = n_uv_rows + 1
        err_u_row = n_uv_rows + 2
        err_v_row = n_uv_rows + 3
        err_spd_row = n_uv_rows + 4
        n_rows = n_uv_rows + 5

        fig = plt.figure(figsize=(6 * 3, 2.8 * n_rows))
        fig.suptitle(
            f"Sample {sample_idx} | Day {int(day * 365):3d}, Hour {int(hour * 365 * 24):2d}h | "
            f"Lat {lat:.2f}, Lon {lon:.2f} — adv-NO rollout +{H1}/{H2}/{H3}d",
            fontsize=13,
            fontweight="bold",
        )
        gs = gridspec.GridSpec(n_rows, 6, figure=fig, wspace=0.35, hspace=0.45)

        for ch in range(n_channels):
            ss = [apply_mask(s1[ch], mask), apply_mask(s2[ch], mask), apply_mask(s3[ch], mask)]
            ch_scale = ss + [apply_mask(g[ch], mask) for g in gt_fields] + [apply_mask(p[ch], mask) for p in pred_fields]
            ch_vmin = np.nanmin(ch_scale)
            ch_vmax = np.nanmax(ch_scale)
            for col, field, title in zip(range(3), ss, ["t[n-2]", "t[n-1]", "t[n]"]):
                ax = fig.add_subplot(gs[2 * ch, col])
                im = ax.imshow(field, vmin=ch_vmin, vmax=ch_vmax, cmap="viridis")
                ax.set_title(f"{CHANNEL_NAMES[ch]} — {title}")
                ax.axis("off")
                plt.colorbar(im, ax=ax, fraction=0.046)

            horizon_pairs = []
            for gf, pf in zip(gt_fields, pred_fields):
                g = apply_mask(gf[ch], mask)
                p = apply_mask(pf[ch], mask)
                vmin = np.nanmin([g, p])
                vmax = np.nanmax([g, p])
                horizon_pairs.append((g, p, vmin, vmax))

            for j, ((g, p, vmin, vmax), title) in enumerate(zip(horizon_pairs, titles)):
                ax = fig.add_subplot(gs[2 * ch, 3 + j])
                im = ax.imshow(g, vmin=vmin, vmax=vmax, cmap="viridis")
                ax.set_title(f"GT {title}")
                ax.axis("off")
                plt.colorbar(im, ax=ax, fraction=0.046)

            for j, ((g, p, vmin, vmax), title, (pe, ue)) in enumerate(zip(horizon_pairs, titles, errs)):
                ax = fig.add_subplot(gs[2 * ch + 1, 3 + j])
                im = ax.imshow(p, vmin=vmin, vmax=vmax, cmap="viridis")
                ax.set_title(f"Pred {title}\nP {pe:.3e} | M {ue:.3e}", fontsize=8)
                ax.axis("off")
                plt.colorbar(im, ax=ax, fraction=0.046)

        ss_spd = [apply_mask(S1, mask), apply_mask(S2, mask), apply_mask(S3, mask)]
        spd_scale = ss_spd + [g for g, _, _, _ in speed_horizon_pairs] + [p for _, p, _, _ in speed_horizon_pairs]
        spd_vmin = np.nanmin(spd_scale)
        spd_vmax = np.nanmax(spd_scale)
        for col, field, title in zip(range(3), ss_spd, ["t[n-2]", "t[n-1]", "t[n]"]):
            ax = fig.add_subplot(gs[speed_row_gt, col])
            im = ax.imshow(field, vmin=spd_vmin, vmax=spd_vmax, cmap="viridis")
            ax.set_title(f"Speed — {title}")
            ax.axis("off")
            plt.colorbar(im, ax=ax, fraction=0.046)
        for j, ((g, p, vmin, vmax), title, (pe, ue)) in enumerate(zip(speed_horizon_pairs, titles, errs)):
            ax = fig.add_subplot(gs[speed_row_gt, 3 + j])
            im = ax.imshow(g, vmin=vmin, vmax=vmax, cmap="viridis")
            ax.set_title(f"GT speed {title}")
            ax.axis("off")
            plt.colorbar(im, ax=ax, fraction=0.046)
            ax = fig.add_subplot(gs[speed_row_pred, 3 + j])
            im = ax.imshow(p, vmin=vmin, vmax=vmax, cmap="viridis")
            ax.set_title(f"Pred speed {title}\nP {pe:.3e} | M {ue:.3e}", fontsize=8)
            ax.axis("off")
            plt.colorbar(im, ax=ax, fraction=0.046)

        for j, (field, title) in enumerate(zip(err_u, titles)):
            ax = fig.add_subplot(gs[err_u_row, 3 + j])
            im = ax.imshow(field, cmap="RdBu_r", vmin=-err_lim_u, vmax=err_lim_u)
            ax.set_title(f"ΔU pred−GT {title}")
            ax.axis("off")
            plt.colorbar(im, ax=ax, fraction=0.046)
        for j, (field, title) in enumerate(zip(err_v, titles)):
            ax = fig.add_subplot(gs[err_v_row, 3 + j])
            im = ax.imshow(field, cmap="RdBu_r", vmin=-err_lim_v, vmax=err_lim_v)
            ax.set_title(f"ΔV pred−GT {title}")
            ax.axis("off")
            plt.colorbar(im, ax=ax, fraction=0.046)
        for j, (field, title) in enumerate(zip(err_spd, titles)):
            ax = fig.add_subplot(gs[err_spd_row, 3 + j])
            im = ax.imshow(field, cmap="RdBu_r", vmin=-err_lim_spd, vmax=err_lim_spd)
            ax.set_title(f"ΔSpeed pred−GT {title}")
            ax.axis("off")
            plt.colorbar(im, ax=ax, fraction=0.046)

        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved to {save_path}")

    return errs[0][0], errs[0][1], errs[1][0], errs[1][1], errs[2][0], errs[2][1]


# ----------------------------
# Usage
# ----------------------------
_sims_shape = np.load(f"{DATA_PREFIX}_sims.npy", mmap_mode="r").shape
CROP_HEIGHT, CROP_WIDTH = _sims_shape[-2], _sims_shape[-1]
N_FRAMES = int(_sims_shape[1])
USE_ZARR_GT = N_FRAMES <= 4

print(f"Run: {RUN_NAME}")
print(f"Model: {MODEL_PATH}")
if BEST_EPOCH:
    print(f"  best val L1 epoch {BEST_EPOCH}: {BEST_VAL_LOSS:.6f}")
print(f"Dataset: {DATA_PREFIX}")
print(f"Crop size: {CROP_HEIGHT} x {CROP_WIDTH} | iD={iD} | frames={N_FRAMES}")
print(f"Rollout horizons: +{H1}d, +{H2}d, +{H3}d (DT={DT}h)")
if USE_ZARR_GT:
    print(f"GT +{H2}d/+{H3}d: loaded from Zarr via export indices")
print(f"Metrics: center {SPEED_PATCH_SIZE}x{SPEED_PATCH_SIZE} and full valid field")

Par = {
    "lb": 3,
    "nf": iD,
    "nx": CROP_HEIGHT,
    "ny": CROP_WIDTH,
    "inp_shift": 0.0,
    "inp_scale": 1.0,
    "out_shift": 0.0,
    "out_scale": 1.0,
    "t_shift": 0.0,
    "t_scale": 1.0,
}
_unet_kwargs = {}
if GENERATOR == "gUnet" and UNET_DROPOUT > 0:
    _unet_kwargs["dropout"] = UNET_DROPOUT
model = Unet2D(
    dim=UNET_DIM,
    Par=Par,
    dim_mults=UNET_DIM_MULTS,
    channels=Par["lb"] * Par["nf"],
    **_unet_kwargs,
)
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu", weights_only=True))

test_data = load_test_data()
(
    s1, s2, s3, s4, s5, s6,
    p1, p2, p3,
    inc_pred1, inc_pred2, inc_pred3,
    inc_gt1, inc_gt2, inc_gt3,
) = run_inference(model, test_data, device=DEVICE, iD=iD)
print("=>", s1.shape, p1.shape, s5.shape)

# Gaussian-circle rollout viz (normalized Δu,Δv)
n_test = int(s1.shape[0])
rng = np.random.default_rng(PLOT_SAMPLE_SEED)
plot_indices = sorted(
    int(i) for i in rng.choice(n_test, size=min(PLOT_N_SAMPLES, n_test), replace=False)
)
print(f"Gaussian-circle plots: {len(plot_indices)} random test idx {plot_indices}")

rollout_horizons = [
    (H1, inc_pred1, inc_gt1, "export"),
    (H2, inc_pred2, inc_gt2, "zarr"),
    (H3, inc_pred3, inc_gt3, "zarr"),
]

for slot, test_idx in enumerate(plot_indices):
    mask_t = test_data["mask"][test_idx]
    day = float(test_data["day"][test_idx].item())
    for horizon_days, pred_batch, gt_batch, gt_src in rollout_horizons:
        pred = pred_batch[test_idx].detach().cpu()
        target = gt_batch[test_idx].detach().cpu()
        pred_plot, plot_names = _uv_with_speed(pred)
        target_plot, _ = _uv_with_speed(target)
        out_path = SAMPLES_GAUSS_DIR / f"sample_{slot:02d}_idx{test_idx}_plus{horizon_days:02d}d.png"
        title = (
            f"test idx {test_idx} | +{horizon_days}d normalized Δu,Δv (GT {gt_src}) | "
            f"day {int(day * 365)} | {RUN_NAME}"
        )
        save_gaussian_circles_sample_figure(
            pred_plot,
            target_plot,
            mask_t,
            out_path,
            gaussian_sigma=GAUSS_SIGMA,
            channel_names=plot_names,
            residual_metric=VIZ_RESIDUAL,
            title=title,
            dpi=150,
            fig_scale=GAUSS_FIG_SCALE,
            circle_radii=GAUSS_CIRCLE_RADII,
            inner_circle_radii=GAUSS_INNER_RADII,
        )
        mae_uv = float(torch.abs(pred[:2] - target[:2])[:, mask_t == 0].mean())
        print(f"Saved {out_path} | masked uv MAE={mae_uv:.4f}")


## Velocity magnitude & persistence error (+15d / +30d / +45d)

Run the inference cell above first (loads +30/+45d GT from Zarr when export has 4 frames).

- **Speed** $|\mathbf{u}| = \sqrt{u^2 + v^2}$
- **Persistence**: RMSE vs anchor speed $t[n]$
- **Model**: RMSE vs GT speed at each horizon

Reported for **center 4×4** and **full valid field**.


In [ ]:
def speed_magnitude(states) -> np.ndarray:
    arr = states.numpy() if torch.is_tensor(states) else np.asarray(states)
    return np.sqrt(arr[:, 0] ** 2 + arr[:, 1] ** 2)


def masked_rmse(field_a: np.ndarray, field_b: np.ndarray, mask: np.ndarray, patch_size: int | None = None) -> float:
    field_a, mask = _crop_metric_region(field_a, mask, patch_size)
    field_b, _ = _crop_metric_region(field_b, mask, patch_size)
    diff2 = (field_b - field_a) ** 2
    valid = mask == 0
    if not np.any(valid):
        return float("nan")
    return float(np.sqrt(diff2[valid].mean()))


def rollout_speed_metrics(
    gt_now,
    gt_h1,
    gt_h2,
    gt_h3,
    pred_h1,
    pred_h2,
    pred_h3,
    mask,
    horizons,
    patch_size: int | None = None,
):
    speed_now = speed_magnitude(gt_now)
    gt_speeds = [speed_magnitude(gt_h1), speed_magnitude(gt_h2), speed_magnitude(gt_h3)]
    pred_speeds = [speed_magnitude(pred_h1), speed_magnitude(pred_h2), speed_magnitude(pred_h3)]
    mask_np = mask.numpy() if torch.is_tensor(mask) else np.asarray(mask)

    rows = []
    for days, gt_spd, pred_spd in zip(horizons, gt_speeds, pred_speeds):
        persist = masked_rmse(speed_now, gt_spd, mask_np, patch_size)
        model = masked_rmse(pred_spd, gt_spd, mask_np, patch_size)
        skill = 1.0 - model / persist if persist > 0 else float("nan")
        rows.append(
            {
                "horizon_days": days,
                "persistence_rmse": persist,
                "model_rmse": model,
                "skill_score": skill,
            }
        )
    return rows


def print_metrics_table(metrics, region_label: str):
    print(f"\nSpeed RMSE (|U,V|) — {region_label}")
    print(f"{'Horizon':>10} {'Persistence':>14} {'Model':>14} {'Skill':>10}")
    print("-" * 52)
    for row in metrics:
        print(
            f"+{row['horizon_days']:>2}d"
            f" {row['persistence_rmse']:14.6f}"
            f" {row['model_rmse']:14.6f}"
            f" {row['skill_score']:10.3f}"
        )


mask_all = test_data["mask"]
metrics_center = rollout_speed_metrics(
    s3, s4, s5, s6, p1, p2, p3, mask_all, horizons=EVAL_HORIZONS, patch_size=SPEED_PATCH_SIZE
)
metrics_full = rollout_speed_metrics(
    s3, s4, s5, s6, p1, p2, p3, mask_all, horizons=EVAL_HORIZONS, patch_size=None
)

print(f"Dataset: {DATA_PREFIX} | N={s3.shape[0]} samples | run: {RUN_NAME}")
print_metrics_table(metrics_center, f"center {SPEED_PATCH_SIZE}x{SPEED_PATCH_SIZE}")
print_metrics_table(metrics_full, "full valid field")

persist_center = [row["persistence_rmse"] for row in metrics_center]
model_center = [row["model_rmse"] for row in metrics_center]
skill_center = [row["skill_score"] for row in metrics_center]
persist_full = [row["persistence_rmse"] for row in metrics_full]
model_full = [row["model_rmse"] for row in metrics_full]
skill_full = [row["skill_score"] for row in metrics_full]

from datetime import datetime, timezone

rollout_results = {
    "export_key": EXPORT_KEY,
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
    "rollout_output_dir": str(TEST_ROLLOUT_DIR),
    "eval_data_prefix": DATA_PREFIX,
    "n_test": int(s3.shape[0]),
    "best_epoch": BEST_EPOCH,
    "best_val_loss": BEST_VAL_LOSS,
    "eval_horizons_days": list(EVAL_HORIZONS),
    "speed_patch_size": SPEED_PATCH_SIZE,
    "metrics_center": metrics_center,
    "metrics_full": metrics_full,
    "evaluated_at": datetime.now(timezone.utc).isoformat(),
}
if USE_SHARED_MODEL:
    rollout_results["model_run_name"] = MODEL_RUN_NAME
    rollout_results["model_run_dir"] = str(MODEL_RUN_DIR)
results_path = save_rollout_results(TEST_ROLLOUT_DIR, rollout_results)
dataset_stats_path = MODEL_RUN_DIR / "dataset_stats.json"
dataset_stats = json.loads(dataset_stats_path.read_text()) if dataset_stats_path.exists() else None
index_path = update_comparison_index(
    MODELS_DIR,
    run_name_value=RUN_NAME,
    export_key=EXPORT_KEY,
    run_dir_path=RUN_DIR,
    dataset_stats=dataset_stats,
    rollout_results=rollout_results,
    training_summary={
        "best_val_loss": BEST_VAL_LOSS,
        "best_epoch": BEST_EPOCH,
    },
    rollout_output_dir=TEST_ROLLOUT_DIR,
)
print(f"Saved rollout results: {results_path}")
print(f"Updated comparison index: {index_path}")


In [ ]:
# Bar plots: center 4x4 and full field
labels = [f"+{d}d" for d in EVAL_HORIZONS]
x = np.arange(len(labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, persist, model, skill, title in [
    (axes[0], persist_center, model_center, skill_center, f"center {SPEED_PATCH_SIZE}x{SPEED_PATCH_SIZE}"),
    (axes[1], persist_full, model_full, skill_full, "full valid field"),
]:
    ax.bar(x - width / 2, persist, width, label="Persistence", color="#ff7f0e")
    ax.bar(x + width / 2, model, width, label="adv-NO model", color="#1f77b4")
    ax.set_xticks(x, labels)
    ax.set_ylabel("Speed RMSE (|U,V|)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    for i, (p, m, s) in enumerate(zip(persist, model, skill)):
        ax.text(i - width / 2, p, f"{p:.3f}", ha="center", va="bottom", fontsize=8)
        ax.text(i + width / 2, m, f"{m:.3f}", ha="center", va="bottom", fontsize=8)
        ax.text(i, max(p, m) * 1.02, f"skill={s:.2f}", ha="center", va="bottom", fontsize=8)

fig.suptitle(f"Rollout speed error — {RUN_NAME}", y=1.02)
plt.tight_layout()
out_path = TEST_ROLLOUT_DIR / "speed_rmse_center_and_full.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")


In [ ]:
# Per-horizon plots: persistence vs model RMSE (+15d / +30d / +45d)
# Row 0 = center 4×4, row 1 = full valid field; one panel per horizon
width = 0.55

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey="row")
region_rows = [
    (persist_center, model_center, skill_center, f"center {SPEED_PATCH_SIZE}×{SPEED_PATCH_SIZE}"),
    (persist_full, model_full, skill_full, "full valid field"),
]

for row, (persist, model, skill, row_title) in enumerate(region_rows):
    for col, (days, p, m, s) in enumerate(zip(EVAL_HORIZONS, persist, model, skill)):
        ax = axes[row, col]
        bars = ax.bar(["Persistence", "Model"], [p, m], color=["#ff7f0e", "#1f77b4"], width=width)
        ax.set_title(f"{row_title} | +{days}d")
        ax.set_ylabel("Speed RMSE (|U,V|)" if col == 0 else "")
        ax.grid(True, axis="y", alpha=0.3)
        for bar, val in zip(bars, [p, m]):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val,
                f"{val:.4f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )
        ax.text(0.5, 0.95, f"skill={s:.3f}", ha="center", va="top", fontsize=9, transform=ax.transAxes)

fig.suptitle(f"Persistence vs model speed RMSE by horizon — {RUN_NAME}", y=1.02, fontsize=13)
plt.tight_layout()
out_path = TEST_ROLLOUT_DIR / "speed_rmse_by_horizon_center_and_full.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

In [ ]:
# Line + point curves: persistence vs model RMSE vs horizon (+15d / +30d / +45d)
horizons = np.array(EVAL_HORIZONS, dtype=float)
xtick_labels = [f"+{int(d)}d" for d in horizons]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, persist, model, title in [
    (axes[0], persist_center, model_center, f"center {SPEED_PATCH_SIZE}×{SPEED_PATCH_SIZE}"),
    (axes[1], persist_full, model_full, "full valid field"),
]:
    ax.plot(
        horizons, persist,
        "-o", color="#ff7f0e", linewidth=2, markersize=9,
        markerfacecolor="#ff7f0e", markeredgecolor="white", markeredgewidth=1.2,
        label="Persistence",
    )
    ax.plot(
        horizons, model,
        "-o", color="#1f77b4", linewidth=2, markersize=9,
        markerfacecolor="#1f77b4", markeredgecolor="white", markeredgewidth=1.2,
        label="adv-NO model",
    )
    for d, p, m in zip(horizons, persist, model):
        ax.annotate(f"{p:.3f}", (d, p), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=8, color="#cc6600")
        ax.annotate(f"{m:.3f}", (d, m), textcoords="offset points", xytext=(0, -14), ha="center", fontsize=8, color="#1a5a9a")

    ax.set_xticks(horizons)
    ax.set_xticklabels(xtick_labels)
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("Speed RMSE (|U,V|)")
    ax.set_title(title)
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Speed RMSE curves — {RUN_NAME}", y=1.02, fontsize=13)
plt.tight_layout()
out_path = TEST_ROLLOUT_DIR / "speed_rmse_curves_center_and_full.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")